# 04 Inference Submission

Thin Kaggle handoff notebook.

This notebook is designed to swap the smoke-test predictor for a Kaggle-safe inference backend while reusing the shared prompt, parsing, and routing modules.

In [ ]:
from pathlib import Path
import json
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    for candidate in [ROOT, *ROOT.parents]:
        if (candidate / "src").exists():
            ROOT = candidate
            break

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT


In [ ]:
from src import load_config
from src.data import load_eval_examples, resolve_path
from src.eval import HeuristicPredictor, evaluate_examples
from src.prompts import get_prompt_variant
from src.solvers import ConservativeRouter

config = load_config(ROOT / "configs/experiments/baseline_eval.yaml")
examples = load_eval_examples(resolve_path(config["data"]["path"]))
sample_examples = examples[: config["kaggle"]["sample_limit"]]
selected_variant = get_prompt_variant(config["selected_prompt_variant"])
router = ConservativeRouter(
    confidence_threshold=config["routing"]["confidence_threshold"]
)
sample_result = evaluate_examples(
    sample_examples,
    prompt_variants=[selected_variant],
    predictor=HeuristicPredictor(),
    router=router if config["routing"]["enabled"] else None,
    run_name="kaggle_handoff_smoke",
    output_dir=ROOT / config["kaggle"]["output_dir"],
)
sample_result.metrics


In [ ]:
note_path = ROOT / config["kaggle"]["output_dir"] / "kaggle_run_note.json"
note_payload = {
    "status": "replace_heuristic_predictor_before_submission",
    "selected_prompt_variant": selected_variant.name,
    "metrics": sample_result.metrics,
    "next_steps": [
        "Attach the real Kaggle dataset.",
        "Replace HeuristicPredictor with the Kaggle inference backend.",
        "Verify boxed answer formatting and runtime budget.",
    ],
}
note_path.write_text(json.dumps(note_payload, indent=2), encoding="utf-8")
print(note_path.read_text(encoding="utf-8"))
